[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/2_Practica/para_casa/S06_01_pandas_eda.ipynb)

# S06 · 01 — Conocer los datos: pandas y exploración (EDA)

Sprint 6 · Módulo M2 (UD 2.1–2.3)

**Contexto TurisData.** El cliente nos ha pasado su histórico de reservas y quiere "meter IA". Antes de entrenar nada hay que **mirar los datos con calma**: qué hay, qué falta, qué distribución tienen y qué relaciones se intuyen. Eso es un *análisis exploratorio* (EDA).

**Este cuaderno tiene 6 ejercicios y un mini-reto.** Los ejercicios 1, 2, 3 y 6 son los mismos que los del cuaderno «en clase» (allí, del 1 al 4).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: solo pulsa ▶ y mira. No hay que escribir nada.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, sigues.
- **Parte B · Explicación** (al final): para cada celda, **qué hace** y **qué resultado debe salir**. Ábrela cuando quieras entender una celda o no te salga. Usa el índice de la izquierda (☰) para saltar.


# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar una tabla

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
ruta = dato("reservas_turisdata.csv")
reservas = pd.read_csv(ruta, parse_dates=["fecha_reserva", "fecha_llegada"])
print(reservas.shape)
reservas.head()

### 👀 A3 · EJEMPLO — Ver los tipos de las columnas

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
reservas.info()

### 👀 A4 · EJEMPLO — Resumir los números

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
# Resumen estadístico de las columnas numéricas
reservas.describe().T.round(2)

### ✏️ A5 · EJERCICIO 1 de 6 — Cargar la tabla

**Tu turno:** Carga `reservas_turisdata.csv` en un DataFrame llamado `df` (con las dos fechas parseadas) y guarda en `n_filas` y `n_columnas` sus dimensiones.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A5.

In [ ]:
# pista: usa pd.read_csv(ruta, parse_dates=[...]) y df.shape
df = ...
n_filas, n_columnas = ...

assert n_filas == 6000, "Revisa la carga: deberían ser 6000 reservas"
assert n_columnas == 18, "Deberían ser 18 columnas"
assert str(df["fecha_llegada"].dtype).startswith("datetime"), "Parsea las fechas con parse_dates=[...]"
print("✔ Ejercicio 1 correcto")

### 👀 A6 · EJEMPLO — Agrupar

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
# Ejemplo: tasa de cancelación por tipo de tarifa
df.groupby("tarifa")["cancelada"].mean().round(3)

### 👀 A7 · EJEMPLO — Agrupar con varias medidas

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
# Ejemplo: varias métricas a la vez
df.groupby("canal").agg(reservas=("id_reserva", "count"),
                        precio_medio=("precio_noche", "mean"),
                        tasa_cancelacion=("cancelada", "mean")).round(2)

### ✏️ A8 · EJERCICIO 2 de 6 — Tasa de cancelación por depósito

**Tu turno:** Calcula `tasa_por_deposito`: la tasa de cancelación (media de `cancelada`) para cada valor de `deposito`, ordenada de mayor a menor.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A8.

In [ ]:
# pista: groupby("deposito")["cancelada"].mean() y sort_values(ascending=False)
tasa_por_deposito = ...

assert tasa_por_deposito.index[-1] == "total", "Con depósito total debería cancelar menos gente. Revisa el orden"
assert tasa_por_deposito.iloc[0] > tasa_por_deposito.iloc[-1], "Ordena de mayor a menor"
print("✔ Ejercicio 2 correcto")

### 👀 A9 · EJEMPLO — Preparar mes y temporadas

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
df["mes_llegada"] = df["fecha_llegada"].dt.month
temporadas = pd.DataFrame({
    "mes_llegada": range(1, 13),
    "temporada": ["alta", "alta", "alta", "baja", "baja", "baja",
                  "alta", "alta", "baja", "baja", "baja", "alta"],
})
temporadas.head(4)

### ✏️ A10 · EJERCICIO 3 de 6 — Unir con las temporadas

**Tu turno:** Une `df` con `temporadas` por `mes_llegada` (unión `left`, para no perder reservas) y guarda el resultado en `df_temp`. Después calcula `tasa_por_temporada`.

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A10.

In [ ]:
# pista: df.merge(temporadas, on="mes_llegada", how="left")
df_temp = ...
tasa_por_temporada = ...      # groupby("temporada") y la media de "cancelada"

assert len(df_temp) == len(df), "Un merge 'left' con clave única no debe cambiar el número de filas"
assert "temporada" in df_temp.columns, "Falta la columna temporada"
assert set(tasa_por_temporada.index) == {"alta", "baja"}
print("✔ Ejercicio 3 correcto")

### 👀 A11 · EJEMPLO — Tres gráficos

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["precio_noche"], bins=30, ax=ejes[0], color="#2a6f97")
ejes[0].set(title="Distribución del precio por noche", xlabel="Precio por noche (€)", ylabel="Reservas")
sns.countplot(data=df, x="canal", ax=ejes[1], color="#2a6f97")
ejes[1].set(title="Reservas por canal", xlabel="Canal", ylabel="Reservas")
ejes[1].tick_params(axis="x", rotation=25)
sns.boxplot(data=df, x="tipo_habitacion", y="precio_noche", ax=ejes[2], color="#8ecae6")
ejes[2].set(title="Precio por tipo de habitación", xlabel="Tipo de habitación", ylabel="Precio por noche (€)")
plt.tight_layout()
plt.show()

### 👀 A12 · EJEMPLO — Serie temporal

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
# Serie temporal: reservas por mes de llegada
por_mes = df.set_index("fecha_llegada").resample("MS")["id_reserva"].count()
plt.figure(figsize=(9, 3.5))
por_mes.plot(marker="o", color="#2a6f97")
plt.title("Reservas por mes de llegada")
plt.xlabel("Mes")
plt.ylabel("Reservas")
plt.show()

### ✏️ A13 · EJERCICIO 4 de 6 — Mediana de precio por tipo de habitación

**Tu turno:** Calcula `mediana_por_tipo`: la mediana de `precio_noche` por `tipo_habitacion`. Luego dibuja un gráfico de barras con ella (título y ejes en español) y responde en la variable `tipo_mas_caro` (texto) cuál es el tipo con mediana más alta.

**Debe salir:** ✔ Ejercicio 4 correcto, y un gráfico de 4 barras

Cambia los `...` y pulsa ▶. Explicación: Parte B, A13.

In [ ]:
# pista: groupby + median; después .plot(kind="bar"); idxmax() te da el más caro
mediana_por_tipo = ...
tipo_mas_caro = ...

assert tipo_mas_caro == "suite", "Revisa: el tipo con mayor mediana de precio debería ser la suite"
assert len(mediana_por_tipo) == 4, "Debe haber 4 tipos de habitación"
print("✔ Ejercicio 4 correcto")

### 👀 A14 · EJEMPLO — Mapa de calor de correlaciones

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
numericas = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
             "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "cancelada"]
corr = df[numericas].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, cbar_kws={"label": "Correlación"})
plt.title("Correlaciones entre variables numéricas")
plt.show()

### ✏️ A15 · EJERCICIO 5 de 6 — Variable más ligada a la cancelación

**Tu turno:** Usando la matriz `corr`, guarda en `var_mas_ligada` el nombre de la variable (distinta de `cancelada`) con **mayor correlación absoluta** con `cancelada`.

**Debe salir:** ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A15.

In [ ]:
# pista: corr["cancelada"].drop("cancelada").abs().idxmax()
var_mas_ligada = ...

assert isinstance(var_mas_ligada, str) and var_mas_ligada != "cancelada", "Debe ser el nombre de otra variable"
assert var_mas_ligada in numericas, "Debe ser una de las variables de la matriz"
print("✔ Ejercicio 5 correcto")

### 👀 A16 · EJEMPLO — Resumen de calidad

Solo pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

In [ ]:
def resumen_calidad(tabla):
    """Devuelve una tabla con el estado de calidad de cada columna."""
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "nulos": tabla.isna().sum(),
        "pct_nulos": (tabla.isna().mean() * 100).round(2),
        "distintos": tabla.nunique(),
    })

resumen_calidad(df)

### ✏️ A17 · EJERCICIO 6 de 6 — Duplicados y nulos de país

**Tu turno:** Calcula `filas_duplicadas` (número de filas repetidas) y `pct_nulos_pais` (porcentaje de nulos de `pais_origen`, redondeado a 1 decimal).

**Debe salir:** ✔ Ejercicio 6 correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A17.

In [ ]:
# pista: df.duplicated().sum() y df["pais_origen"].isna().mean() * 100
filas_duplicadas = ...
pct_nulos_pais = ...

assert filas_duplicadas == 0, "Esta tabla limpia no debería tener duplicados"
assert 1.0 <= pct_nulos_pais <= 3.0, "El porcentaje de nulos de pais_origen debería rondar el 2 %"
print("✔ Ejercicio 6 correcto")

### ✏️ A18 · MINI-RETO — Tres hallazgos para el cliente

**Tu turno:** Escribe en la lista `hallazgos` **tres frases** (texto) que un gerente de hotel entendería, apoyadas en lo que has visto (por ejemplo, sobre canales, depósitos, precios o temporada). Cada frase debe incluir al menos **una cifra** que hayas calculado.

**Debe salir:** ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶. Explicación: Parte B, A18.

In [ ]:
# pista: frases del estilo "Con depósito total cancela el X % frente al Y % sin depósito"
hallazgos = [
    "...",   # frase 1, con una cifra
    "...",   # frase 2, con una cifra
    "...",   # frase 3, con una cifra
]

assert len(hallazgos) >= 3, "Escribe al menos tres hallazgos"
assert all(isinstance(h, str) and any(c.isdigit() for c in h) for h in hallazgos), "Cada hallazgo debe ser texto e incluir una cifra"
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

## 1. (sin código) Qué hace un Data Scientist y qué tipos de dato manejamos

Un Data Scientist convierte preguntas de negocio en datos, modelos y recomendaciones.
Los datos pueden ser **estructurados** (tablas: lo de hoy), **no estructurados**
(texto, imágenes, audio) o **series temporales** (valores ordenados en el tiempo).
Dentro de una tabla, cada columna es de un tipo que condiciona qué podemos hacer con ella:

| Tipo | Ejemplo en TurisData | Gráfico habitual |
|---|---|---|
| Numérica continua | `precio_noche` | histograma, caja |
| Numérica discreta | `noches`, `adultos` | barras, histograma |
| Categórica | `canal`, `tipo_habitacion` | barras |
| Fecha | `fecha_llegada` | línea temporal |
| Objetivo (0/1) | `cancelada` | tasa por grupo |

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `pandas` (tablas), `numpy` (números), `matplotlib` y `seaborn` (gráficos).
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - `SEMILLA = 42` fija el azar para que todos obtengáis lo mismo.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 2. Cargar e inspeccionar

#### A2 · Ejemplo — Cargar una tabla

Lo primero siempre es: ¿cuántas filas y columnas? ¿de qué tipo es cada una? ¿qué pinta tienen los primeros registros? Se usa `pd.read_csv(..., parse_dates=[...])` para que las fechas se lean como fechas y no como texto.

- **Qué hace:**
    - `dato("reservas_turisdata.csv")` devuelve dónde está el archivo y se guarda en `ruta`.
    - `pd.read_csv(ruta, parse_dates=[...])` lee el CSV como tabla (*DataFrame*) llamada `reservas`, con `fecha_reserva` y `fecha_llegada` como fechas.
    - `print(reservas.shape)` escribe `(filas, columnas)` y `reservas.head()` enseña las 5 primeras filas.
- **Qué debe salir:**
    - `(6000, 18)` y una tabla de 5 filas.
- **Cómo leerlo:**
    - Cada fila es **una reserva** y cada columna, un dato de ella.
    - Primera fila de ejemplo: reserva 5760, hecha el 2024-01-01 para llegar el 2024-03-21 (80 días de antelación), 3 noches, 3 adultos y 2 niños, habitación `estandar`, canal `empresa`, tarifa `flexible`, 89,24 € por noche y `cancelada` = 1 (se canceló).

#### A3 · Ejemplo — Ver los tipos de las columnas

- **Qué hace:**
    - `reservas.info()` lista cada columna con su **tipo** y cuántos valores **no nulos** tiene.
- **Qué debe salir:**
    - 6000 entradas y 18 columnas: 2 de fecha (`datetime64`), 9 enteros (`int64`), 1 decimal (`precio_noche`, `float64`) y 6 de texto (`object`, o `str` según la versión de pandas).
    - `pais_origen` tiene 5862 no nulos.
- **Cómo leerlo:**
    - Una columna con menos «non-null» que el total **tiene huecos**: a `pais_origen` le faltan 138 valores (6000 − 5862).

#### A4 · Ejemplo — Resumir los números

- **Qué hace:**
    - `describe()` resume las columnas numéricas: recuento, media, desviación, mínimo, cuartiles y máximo. `.T` la gira (una fila por columna) y `.round(2)` deja 2 decimales.
- **Qué debe salir:**
    - Una tabla con una fila por cada columna numérica (10).
- **Cómo leerlo:**
    - `noches`: media 5,52 (de 1 a 15).
    - `adultos`: media 2,2.
    - `precio_noche`: media 102,87 € (de 52 € a 249,35 €).
    - `antelacion_dias`: media 89,9 (de 1 a 400 días).
    - `cancelada`: media 0,459 → **el 45,9 % de las reservas se cancela**.

#### A5 · Ejercicio 1 — Cargar la tabla

- **Qué hace:**
    - Es la misma carga del ejemplo anterior, pero guardada en `df` (el nombre que usa el resto del cuaderno).
    - `df.shape` devuelve dos números; `n_filas, n_columnas = ...` los reparte en dos variables.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto` (6000 filas y 18 columnas).
- **Cómo leerlo:**
    - Desde aquí la tabla se llama `df`. La del ejemplo (`reservas`) ya no se usa.
- **Si falla:**
    - `Revisa la carga: deberían ser 6000 reservas` → `df` no se ha cargado bien.
    - `Parsea las fechas con parse_dates=[...]` → falta `parse_dates` o algún nombre está mal escrito.
    - `NameError: name 'ruta' is not defined` → no has ejecutado el primer ejemplo de esta sección.

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
df = pd.read_csv(ruta, parse_dates=["fecha_reserva", "fecha_llegada"])
n_filas, n_columnas = df.shape
```

</details>

## 3. Filtrar y agrupar

#### A6 · Ejemplo — Agrupar

Se filtra con condiciones (`df[df["canal"] == "directo"]`) y se resume por grupos con `groupby`. Como `cancelada` vale 0 o 1, **su media es la tasa de cancelación**.

- **Qué hace:**
    - `groupby("tarifa")` separa las reservas por tarifa.
    - `["cancelada"].mean()` calcula la media de `cancelada` en cada grupo: la tasa de cancelación.
    - `round(3)` deja 3 decimales.
- **Qué debe salir:**
    - `flexible` 0.519 · `no_reembolsable` 0.367.
- **Cómo leerlo:**
    - Con tarifa flexible se cancela el 51,9 % de las reservas; con tarifa no reembolsable, el 36,7 %.

#### A7 · Ejemplo — Agrupar con varias medidas

- **Qué hace:**
    - `agg(nombre=(columna, función), ...)` calcula varias medidas a la vez: cuántas reservas, el precio medio y la tasa de cancelación de cada canal.
- **Qué debe salir:**
    - Una tabla de 4 filas (un canal en cada una) y 3 columnas.
- **Cómo leerlo:**
    - `agencia`: 1190 reservas · `directo`: 1462 · `empresa`: 328 · `web_intermediaria`: 3020 (la mitad del total).
    - El **precio medio es casi igual** en los cuatro (entre 101,53 € y 103,89 €), pero la **tasa de cancelación cambia mucho**: `directo` 0,31 frente a `web_intermediaria` 0,54.

#### A8 · Ejercicio 2 — Tasa de cancelación por depósito

- **Qué hace:**
    - Repite el ejemplo de agrupar cambiando la columna: `"tarifa"` → `"deposito"`, y añade `sort_values(ascending=False)` para ordenar de mayor a menor.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. Si escribes `tasa_por_deposito`: `sin_deposito` 0.540 · `parcial` 0.409 · `total` 0.264.
- **Cómo leerlo:**
    - Sin depósito se cancela el 54 % de las reservas; con depósito total, solo el 26,4 %: **pagar por adelantado va unido a menos cancelaciones** (asociación, no prueba de causa).
- **Si falla:**
    - `Con depósito total debería cancelar menos gente. Revisa el orden` → falta el orden, o está al revés.
    - `Ordena de mayor a menor` → falta `ascending=False`.
    - `KeyError` → el nombre de la columna está mal escrito.

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
tasa_por_deposito = df.groupby("deposito")["cancelada"].mean().sort_values(ascending=False)
```

</details>

## 4. Unir tablas (`merge`)

#### A9 · Ejemplo — Preparar mes y temporadas

A menudo la información está repartida en varias tablas. `merge` las une por una columna común, como un `JOIN` de SQL. Aquí se añade a cada reserva la **temporada** según el mes de llegada, con una pequeña tabla de equivalencias.

- **Qué hace:**
    - `df["fecha_llegada"].dt.month` saca el número de mes (1 a 12) de cada fecha y lo guarda en una columna nueva, `mes_llegada`.
    - `temporadas` es una tabla de 12 filas: temporada **alta** en enero, febrero, marzo, julio, agosto y diciembre; **baja** en el resto.
- **Qué debe salir:**
    - Una tabla con los meses 1 a 4: los tres primeros `alta` y el cuarto `baja`.
- **Cómo leerlo:**
    - Esta celda **añade una columna a `df`**. Si la ejecutas dos veces no pasa nada (se sobrescribe).

#### A10 · Ejercicio 3 — Unir con las temporadas

- **Qué hace:**
    - `df.merge(temporadas, on="mes_llegada", how="left")` añade a cada reserva su temporada. `how="left"` conserva **todas** las reservas de `df`.
    - Después se agrupa por la columna nueva `temporada` (como en la sección 3).
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. Si escribes `tasa_por_temporada`: `alta` 0.479 · `baja` 0.438.
- **Cómo leerlo:**
    - Se cancela algo más en temporada alta (47,9 %) que en baja (43,8 %): unos 4 puntos de diferencia.
    - Tras un `merge` con `left`, el número de filas **no debe cambiar**: es la comprobación más rápida de que salió bien.
- **Si falla:**
    - `Un merge 'left' ... no debe cambiar el número de filas` → la clave no es `mes_llegada` o falta `how="left"`.
    - `Falta la columna temporada` → has unido las tablas al revés o no has guardado el resultado en `df_temp`.
    - `KeyError: 'mes_llegada'` → no has ejecutado la celda anterior.

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
df_temp = df.merge(temporadas, on="mes_llegada", how="left")
tasa_por_temporada = df_temp.groupby("temporada")["cancelada"].mean()
```

</details>

## 5. Visualización exploratoria: el gráfico según el dato

#### A11 · Ejemplo — Tres gráficos

El gráfico adecuado depende del tipo de dato:

- **Un número** → histograma. ¿Simétrico? ¿Con cola? ¿Con valores raros?
- **Una categoría** → barras de recuento.
- **Un número por categoría** → diagrama de caja (*boxplot*).
- **Dos números** → dispersión.
- **Una fecha** → línea en el tiempo.
- **Muchos números a la vez** → mapa de calor de correlaciones.

- **Qué hace:**
    - `plt.subplots(1, 3)` prepara una fila con tres huecos para gráficos.
    - `histplot` (histograma), `countplot` (barras de recuento) y `boxplot` (diagrama de caja) dibujan uno en cada hueco.
    - `.set(title=..., xlabel=..., ylabel=...)` pone título y nombres de ejes **en español**.
- **Qué debe salir:**
    - Una figura con tres gráficos.
- **Cómo leerlo:**
    - **Histograma:** la mayoría de las reservas cuesta entre unos 78 € y 119 €, con una cola hacia los 250 €.
    - **Barras:** `web_intermediaria` aporta 3020 de las 6000 reservas (la mitad).
    - **Diagrama de caja:** la raya central es la mediana; la caja es el 50 % central de los precios; los puntos sueltos son valores atípicos. La caja de la `suite` (mediana ≈ 175 €) está muy por encima de la `estandar` (≈ 78 €).

#### A12 · Ejemplo — Serie temporal

- **Qué hace:**
    - `resample("MS")` agrupa por mes (el inicio de cada mes) y `count()` cuenta las reservas de cada uno.
    - `por_mes.plot(marker="o")` dibuja la línea con un punto por mes.
- **Qué debe salir:**
    - Una línea de 36 puntos (de enero de 2024 a diciembre de 2026).
- **Cómo leerlo:**
    - La línea **sube** de 17 reservas (enero de 2024) a unas 250 al mes, y **cae al final** (2026).
    - No significa que el hotel venda menos: la tabla recoge reservas hechas entre 2024 y 2025, así que al principio aún no habían llegado los clientes y al final solo quedan las llegadas más lejanas. **Los extremos de una serie temporal engañan.**

#### A13 · Ejercicio 4 — Mediana de precio por tipo de habitación

- **Qué hace:**
    - Agrupa como en la sección 3, pero con `median()` (la mediana) en vez de `mean()`.
    - Dibuja las barras con `.plot(kind="bar")`, con título y ejes en español.
    - `idxmax()` da el **nombre** del grupo con el valor más alto.
- **Qué debe salir:**
    - `✔ Ejercicio 4 correcto` y 4 barras de menor a mayor: `estandar` 77,6 € · `superior` 105,7 € · `familiar` 123,0 € · `suite` 175,5 €.
    - `tipo_mas_caro` vale `"suite"`.
- **Cómo leerlo:**
    - La **mediana** es el valor del medio: no se deja arrastrar por unos pocos precios muy altos, como sí hace la media.
- **Si falla:**
    - `Revisa: el tipo con mayor mediana...` → has usado `mean()`, o `tipo_mas_caro` no es el nombre del tipo (usa `idxmax()`).
    - `Debe haber 4 tipos de habitación` → agrupa por `"tipo_habitacion"`.

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
mediana_por_tipo = df.groupby("tipo_habitacion")["precio_noche"].median().sort_values()
mediana_por_tipo.plot(kind="bar", color="#2a6f97")
plt.title("Precio mediano por tipo de habitación")
plt.xlabel("Tipo de habitación")
plt.ylabel("Precio mediano por noche (€)")
plt.show()
tipo_mas_caro = mediana_por_tipo.idxmax()
```

</details>

## 6. Relaciones entre variables numéricas

#### A14 · Ejemplo — Mapa de calor de correlaciones

La **correlación** (entre −1 y 1) mide cuánto se mueven juntas dos variables. Ojo: correlación **no** es causalidad, y solo detecta relaciones «en línea recta».

- **Qué hace:**
    - `df[numericas].corr()` calcula la correlación de cada pareja de variables: una matriz de 9 × 9 que se guarda en `corr`.
    - `sns.heatmap(..., annot=True)` la dibuja como mapa de calor con los números dentro: cuanto más intenso el color, más relación.
- **Qué debe salir:**
    - Un cuadrado de 9 × 9, con unos (1.00) en la diagonal.
- **Cómo leerlo:**
    - Mira la fila (o columna) de `cancelada`: `antelacion_dias` tiene **0,28**, la más alta; el resto está entre −0,07 y 0,04, casi sin relación.
    - Lectura: cuanta más antelación tiene una reserva, **algo más probable** es que se cancele.

#### A15 · Ejercicio 5 — Variable más ligada a la cancelación

- **Qué hace:**
    - `corr["cancelada"]` es la columna de la matriz con las correlaciones con `cancelada`.
    - `.drop("cancelada")` quita la correlación consigo misma (que siempre es 1).
    - `.abs()` quita el signo (−0,07 y 0,07 cuentan igual) y `.idxmax()` da el **nombre** del valor más alto.
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto`. `var_mas_ligada` vale `"antelacion_dias"`.
- **Cómo leerlo:**
    - Sin `.abs()`, una correlación muy negativa quedaría la última aunque fuera muy fuerte.
- **Si falla:**
    - `Debe ser el nombre de otra variable` → falta `.drop("cancelada")`: sale `cancelada`.
    - `NameError: name 'corr' is not defined` → no has ejecutado la celda del mapa de calor.

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
var_mas_ligada = corr["cancelada"].drop("cancelada").abs().idxmax()
```

</details>

## 7. Calidad del dato: primer chequeo

#### A16 · Ejemplo — Resumen de calidad

Antes de modelar hay que saber **cuánto podemos fiarnos** de la tabla. Un buen resumen de calidad recoge, por columna: tipo, nº de nulos, % de nulos y nº de valores distintos.

- **Qué hace:**
    - `def resumen_calidad(tabla):` crea una **función**: un trozo de código con nombre que se puede usar con cualquier tabla.
    - Dentro monta una tabla con, por columna: el tipo, cuántos nulos tiene, el % de nulos y cuántos valores distintos.
    - `resumen_calidad(df)` la usa con nuestra tabla.
- **Qué debe salir:**
    - Una tabla de 19 filas (las 18 columnas más `mes_llegada`).
- **Cómo leerlo:**
    - **Nulos:** solo `pais_origen` tiene huecos: 138 (2,3 %). Todas las demás columnas están completas.
    - **Distintos:** `id_reserva` tiene 6000 (una por reserva, sin repetidas). `canal` tiene 4, `tarifa` 2, `deposito` 3 y `pais_origen` 8: pocas categorías, bien escritas.
    - Esta tabla está **limpia**. La del reto (`reservas_sucio.csv`) no lo estará.

#### A17 · Ejercicio 6 — Duplicados y nulos de país

- **Qué hace:**
    - `df.duplicated()` marca cada fila repetida con verdadero; `.sum()` cuenta los verdaderos.
    - `df["pais_origen"].isna()` marca los huecos; `.mean()` da la proporción (los verdaderos valen 1) y `* 100` la pasa a %; `round(..., 1)` deja un decimal.
- **Qué debe salir:**
    - `✔ Ejercicio 6 correcto`. `filas_duplicadas` vale 0 y `pct_nulos_pais` vale 2.3.
- **Cómo leerlo:**
    - Mismo truco que con `cancelada`: la media de verdadero/falso es una proporción.
- **Si falla:**
    - `El porcentaje de nulos ... debería rondar el 2 %` → se te ha olvidado `* 100` (sale 0.023).

<details><summary>Ver la solución (solo después de intentarlo)</summary>

```python
filas_duplicadas = int(df.duplicated().sum())
pct_nulos_pais = round(df["pais_origen"].isna().mean() * 100, 1)
```

</details>

## 8. Mini-reto: tres hallazgos para el cliente

#### A18 · Mini-reto — Tres hallazgos para el cliente

- **Qué hace:**
    - Una lista de Python con tres textos. Cada texto es una frase para el gerente.
    - Una buena frase tiene: **una cifra**, **una comparación** y **qué significa**.
- **Qué debe salir:**
    - `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - Ejemplos de frases válidas con las cifras de este cuaderno:
    - «Con depósito total cancela el 26 % de las reservas frente al 54 % sin depósito.»
    - «La suite tiene el precio mediano más alto (175 € por noche).»
    - «Las reservas en temporada alta cancelan un 48 % frente a un 44 % en baja.»
    - Mejor con tus propias cifras: calcula otra cosa (por ejemplo, la tasa por canal) y úsala.
- **Si falla:**
    - `Cada hallazgo debe ser texto e incluir una cifra` → alguna frase no lleva ningún número, o sigue con `"..."`.

## 9. Para reflexionar

1. ¿Qué gráfico usarías para comparar el precio entre canales? ¿Y para ver si `antelacion_dias` se relaciona con cancelar?
2. Una correlación de 0,05 entre dos variables, ¿significa que no hay relación?
3. ¿Por qué usar la media de `cancelada` como tasa es cómodo? ¿Qué pasaría si `cancelada` estuviera codificada como "sí"/"no"?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Entre canales: un **diagrama de caja** del precio por canal. Para `antelacion_dias` frente a cancelar: una **dispersión** o una caja de `antelacion_dias` según `cancelada`.
2. **No.** La correlación solo detecta relaciones «en línea recta»: con 0,05 casi no hay relación lineal, pero podría haber una curva.
3. Porque con 0 y 1 la **media es directamente la proporción** de canceladas. Con «sí»/«no» no se puede hacer la media: habría que pasarlo antes a 0/1 o contar cada valor.

</details>

## 10. Resumen: qué has aprendido
- Inspeccionar: `shape`, `info`, `describe`, `head`.
- Filtrar, agrupar (`groupby`) y unir (`merge`).
- Elegir el gráfico según el tipo de dato.
- Un resumen de calidad por columna es el primer paso antes de limpiar (siguiente práctica).